In [0]:
# Partial-safe Blob 3 alternative.
# Merges only durable Blob 2 outputs. If worklist rows remain, it preserves the
# checkpoint at source_start_version - 1, then removes this run's staging artifacts.
# The next Blob 1 run rereads the incomplete source version and its target anti-join
# selects only rows not merged here.

from datetime import datetime, timezone
import json
import re

from delta.tables import DeltaTable
from pyspark.sql import Row, functions as F


dbutils.widgets.text("RUN_ID", "")
dbutils.widgets.text("METADATA_TABLE", "4_prod.tmp.blob_pipeline_metadata_v3")
dbutils.widgets.text("CHECKPOINT_TABLE", "4_prod.tmp.blob_pipeline_checkpoint_v3")
dbutils.widgets.text("HISTORY_TABLE", "4_prod.logs.mill_blob_extractor_history")
dbutils.widgets.text("METRICS_TABLE", "4_prod.logs.mill_blob_metrics")
dbutils.widgets.text("PROCESSING_CONFIG_TABLE", "4_prod.tmp.blob_processing_config_v3")
dbutils.widgets.dropdown("CLEANUP", "true", ["false", "true"])

In [0]:


RUN_ID = dbutils.widgets.get("RUN_ID").strip()
METADATA_TABLE = dbutils.widgets.get("METADATA_TABLE").strip()
CHECKPOINT_TABLE = dbutils.widgets.get("CHECKPOINT_TABLE").strip()
HISTORY_TABLE = dbutils.widgets.get("HISTORY_TABLE").strip()
METRICS_TABLE = dbutils.widgets.get("METRICS_TABLE").strip()
PROCESSING_CONFIG_TABLE = dbutils.widgets.get("PROCESSING_CONFIG_TABLE").strip()
CLEANUP = dbutils.widgets.get("CLEANUP").lower() == "true"

if not RUN_ID or not re.fullmatch(r"[A-Za-z0-9_.:-]+", RUN_ID):
    raise ValueError("A safe, explicit RUN_ID is required")


def validate_table_name(value: str) -> str:
    if not re.fullmatch(r"[A-Za-z0-9_]+(?:\.[A-Za-z0-9_]+){1,2}", value):
        raise ValueError(f"Unsafe table identifier: {value!r}")
    return value


def quote_table(value: str) -> str:
    validate_table_name(value)
    tick = chr(96)
    return ".".join(f"{tick}{part}{tick}" for part in value.split("."))


for table_name in (
    METADATA_TABLE, CHECKPOINT_TABLE, HISTORY_TABLE, METRICS_TABLE,
    PROCESSING_CONFIG_TABLE,
):
    validate_table_name(table_name)


def sql_string(value: str) -> str:
    return "'" + (value or "").replace("'", "''") + "'"


def get_metadata():
    rows = spark.table(METADATA_TABLE).filter(F.col("run_id") == RUN_ID).limit(2).collect()
    if len(rows) != 1:
        raise RuntimeError(f"Expected one metadata row for {RUN_ID}; found {len(rows)}")
    return rows[0]


def parse_tables(value: str):
    tables = [item.strip() for item in (value or "").split(",") if item.strip()]
    for table in tables:
        validate_table_name(table)
    return tables


def union_tables(tables):
    frames = [spark.table(table) for table in tables]
    if not frames:
        raise RuntimeError("No staging tables supplied")
    result = frames[0]
    for frame in frames[1:]:
        result = result.unionByName(frame, allowMissingColumns=True)
    return result


def align_to_table(frame, target_table):
    expressions = []
    for field in spark.table(target_table).schema:
        if field.name in frame.columns:
            expressions.append(F.col(field.name).cast(field.dataType).alias(field.name))
        else:
            expressions.append(F.lit(None).cast(field.dataType).alias(field.name))
    return frame.select(*expressions)


def assert_unique(frame, keys, label):
    duplicates = frame.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count()
    if duplicates:
        raise RuntimeError(f"{label} contains duplicate merge keys {keys}")


def merge_all_columns(source, target_table, condition, preserve_on_null=()):
    target_columns = [field.name for field in spark.table(target_table).schema]
    update_map = {}
    insert_map = {}
    tick = chr(96)
    for column in target_columns:
        quoted = f"{tick}{column}{tick}"
        insert_map[column] = f"s.{quoted}"
        if column in preserve_on_null:
            update_map[column] = f"coalesce(s.{quoted}, t.{quoted})"
        else:
            update_map[column] = f"s.{quoted}"
    (
        DeltaTable.forName(spark, target_table)
        .alias("t")
        .merge(source.alias("s"), condition)
        .whenMatchedUpdate(set=update_map)
        .whenNotMatchedInsert(values=insert_map)
        .execute()
    )


def set_checkpoint(version, pipeline_id, source_table, trust_filter):
    checkpoint_rows = (
        spark.table(CHECKPOINT_TABLE)
        .filter(
            (F.col("pipeline_id") == pipeline_id)
            & (F.col("source_table") == source_table)
            & (F.col("trust_filter") == trust_filter)
        )
        .orderBy(F.col("updated_ts").desc_nulls_last())
        .limit(1)
        .collect()
    )
    if checkpoint_rows:
        existing = int(checkpoint_rows[0]["last_source_version"])
        if existing > version:
            raise RuntimeError(
                f"Existing checkpoint {existing} is ahead of safe checkpoint {version}; "
                "refusing to finalize an overlapping stale run"
            )
        version = max(existing, version)

    schema = spark.table(CHECKPOINT_TABLE).schema
    checkpoint_source = spark.createDataFrame([
        Row(
            pipeline_id=pipeline_id,
            source_table=source_table,
            trust_filter=trust_filter,
            last_source_version=int(version),
            last_run_id=RUN_ID,
            updated_ts=datetime.now(timezone.utc).replace(tzinfo=None),
        )
    ], schema=schema)
    (
        DeltaTable.forName(spark, CHECKPOINT_TABLE)
        .alias("t")
        .merge(
            checkpoint_source.alias("s"),
            "t.pipeline_id = s.pipeline_id AND t.source_table = s.source_table "
            "AND t.trust_filter = s.trust_filter",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
meta = get_metadata()
if meta["status"] in {"complete", "partial_complete"}:
    dbutils.notebook.exit(json.dumps({
        "status": "already_complete",
        "run_id": RUN_ID,
        "metadata_status": meta["status"],
    }))
if meta["status"] not in {
    "processing_partial", "processing_complete", "partial_merging",
    "partial_merge_failed", "merge_failed",
}:
    raise RuntimeError(f"Run {RUN_ID} is in non-finalizable status {meta['status']!r}")

WORKLIST_TABLE = meta["worklist_table"]
TARGET_TABLE = meta["target_table"]
PIPELINE_ID = meta["pipeline_id"]
SOURCE_TABLE = meta["source_table"]
TRUST_FILTER = meta["trust_filter"]
SOURCE_START_VERSION = int(meta["source_start_version"])
SOURCE_END_VERSION = int(meta["source_end_version"])
TOTAL_EVENTS = int(meta["total_events"] or 0)
batch_tables = parse_tables(meta["batch_tables"])
history_tables = parse_tables(meta["history_tables"])

required_tables = [WORKLIST_TABLE] + batch_tables + history_tables
missing_tables = [
    table for table in required_tables
    if not table or not spark.catalog.tableExists(table)
]
if missing_tables:
    raise RuntimeError(f"Run {RUN_ID} is missing recoverability artifacts: {missing_tables}")

worklist = spark.table(WORKLIST_TABLE)
completed_worklist_count = int(worklist.filter(F.col("status") == "completed").count())
remaining_worklist_count = int(worklist.filter(F.col("status") != "completed").count())
is_partial = remaining_worklist_count > 0

spark.sql(f"""
  UPDATE {quote_table(METADATA_TABLE)}
  SET status = 'partial_merging', error_message = NULL
  WHERE run_id = {sql_string(RUN_ID)}
""")


In [0]:
try:
    batch = union_tables(batch_tables)
    history = union_tables(history_tables)
    batch_count = int(batch.count())
    history_count = int(history.count())

    if batch_count <= 0:
        raise RuntimeError("No completed staging rows are available to merge")
    if batch_count != history_count or batch_count != completed_worklist_count:
        raise RuntimeError(
            "Completed staging counts are inconsistent: "
            f"worklist_completed={completed_worklist_count}, "
            f"batch={batch_count}, history={history_count}, "
            f"worklist_remaining={remaining_worklist_count}"
        )

    assert_unique(batch, ["EVENT_ID", "ADC_UPDT"], "batch source")
    assert_unique(
        history,
        ["run_id", "EVENT_ID", "ADC_UPDT", "source"],
        "history source",
    )

    batch_keys = batch.select("EVENT_ID", "ADC_UPDT").distinct()
    if int(batch_keys.count()) != batch_count:
        raise RuntimeError("Batch key count does not match batch row count")

    bronze_source = align_to_table(batch.drop("metrics"), TARGET_TABLE)
    merge_all_columns(
        bronze_source,
        TARGET_TABLE,
        "t.EVENT_ID = s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT",
        preserve_on_null=("anon_text",),
    )

    history_source = history
    if "extracted_date" not in history_source.columns:
        history_source = history_source.withColumn("extracted_date", F.current_date())
    history_source = align_to_table(history_source, HISTORY_TABLE)
    merge_all_columns(
        history_source,
        HISTORY_TABLE,
        "t.run_id = s.run_id AND t.EVENT_ID = s.EVENT_ID "
        "AND t.ADC_UPDT <=> s.ADC_UPDT AND t.source <=> s.source",
    )

    metrics_source = batch.select(
        "EVENT_ID",
        F.lit(RUN_ID).alias("JOB_ID"),
        "ADC_UPDT",
        "STATUS",
        "metrics",
        F.current_timestamp().alias("process_ts"),
    )
    metrics_source = align_to_table(metrics_source, METRICS_TABLE)
    assert_unique(
        metrics_source,
        ["JOB_ID", "EVENT_ID", "ADC_UPDT"],
        "metrics source",
    )
    merge_all_columns(
        metrics_source,
        METRICS_TABLE,
        "t.JOB_ID = s.JOB_ID AND t.EVENT_ID = s.EVENT_ID "
        "AND t.ADC_UPDT <=> s.ADC_UPDT",
    )

    target_matches = batch_keys.alias("b").join(
        spark.table(TARGET_TABLE).select("EVENT_ID", "ADC_UPDT").alias("t"),
        (F.col("b.EVENT_ID") == F.col("t.EVENT_ID"))
        & F.col("b.ADC_UPDT").eqNullSafe(F.col("t.ADC_UPDT")),
        "inner",
    ).select("b.EVENT_ID", "b.ADC_UPDT")
    matched_versions = int(target_matches.distinct().count())
    duplicate_target_versions = int(
        target_matches.groupBy("EVENT_ID", "ADC_UPDT")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    history_versions = int(
        spark.table(HISTORY_TABLE)
        .filter((F.col("run_id") == RUN_ID) & (F.col("source") == "forward"))
        .select("EVENT_ID", "ADC_UPDT")
        .distinct()
        .count()
    )
    metrics_versions = int(
        spark.table(METRICS_TABLE)
        .filter(F.col("JOB_ID") == RUN_ID)
        .select("EVENT_ID", "ADC_UPDT")
        .distinct()
        .count()
    )

    validation = {
        "merged_expected": batch_count,
        "target_versions": matched_versions,
        "duplicate_target_versions": duplicate_target_versions,
        "history_versions": history_versions,
        "metrics_versions": metrics_versions,
        "unmerged_worklist_rows": remaining_worklist_count,
    }
    if (
        matched_versions != batch_count
        or duplicate_target_versions != 0
        or history_versions != batch_count
        or metrics_versions != batch_count
    ):
        raise RuntimeError(
            "Post-merge validation failed: "
            + json.dumps(validation, sort_keys=True)
        )

    # A partial merge must not checkpoint the incomplete source commit. Keep the
    # cursor immediately before this run so Blob 1 rereads it and anti-joins rows
    # successfully merged above.
    safe_checkpoint = (
        SOURCE_START_VERSION - 1 if is_partial else SOURCE_END_VERSION
    )
    set_checkpoint(
        safe_checkpoint,
        PIPELINE_ID,
        SOURCE_TABLE,
        TRUST_FILTER,
    )

    final_status = "partial_complete" if is_partial else "complete"
    spark.sql(f"""
      UPDATE {quote_table(METADATA_TABLE)}
      SET status = {sql_string(final_status)},
          merged_events = {batch_count},
          merged_ts = current_timestamp(),
          error_message = NULL
      WHERE run_id = {sql_string(RUN_ID)}
    """)

except Exception as exc:
    message = f"{type(exc).__name__}: {str(exc)[:8000]}"
    spark.sql(f"""
      UPDATE {quote_table(METADATA_TABLE)}
      SET status = 'partial_merge_failed',
          error_message = {sql_string(message)}
      WHERE run_id = {sql_string(RUN_ID)}
    """)
    raise


In [0]:
cleanup_warnings = []
if CLEANUP:
    for table in [WORKLIST_TABLE] + batch_tables + history_tables:
        try:
            spark.sql(f"DROP TABLE IF EXISTS {quote_table(table)}")
        except Exception as exc:
            cleanup_warnings.append(
                f"{table}: {type(exc).__name__}: {str(exc)[:300]}"
            )
    if spark.catalog.tableExists(PROCESSING_CONFIG_TABLE):
        try:
            spark.sql(f"""
              DELETE FROM {quote_table(PROCESSING_CONFIG_TABLE)}
              WHERE run_id = {sql_string(RUN_ID)}
            """)
        except Exception as exc:
            cleanup_warnings.append(
                f"{PROCESSING_CONFIG_TABLE} row: "
                f"{type(exc).__name__}: {str(exc)[:300]}"
            )

result = {
    "status": "partial_success" if is_partial else "success",
    "run_id": RUN_ID,
    "merged_events": batch_count,
    "destaged_events": remaining_worklist_count,
    "total_events": TOTAL_EVENTS,
    "checkpoint_version": SOURCE_START_VERSION - 1 if is_partial else SOURCE_END_VERSION,
    "next_blob1_start_version": SOURCE_START_VERSION if is_partial else SOURCE_END_VERSION + 1,
    "cleanup_warnings": cleanup_warnings,
}
dbutils.jobs.taskValues.set(
    key="MERGE_RESULT",
    value=json.dumps(result, sort_keys=True),
)
dbutils.notebook.exit(json.dumps(result, sort_keys=True))